In [4]:
# run first time
import sys
!{sys.executable} -m pip install "gymnasium[box2d]"==1.2.3
# !{sys.executable} -m pip install "gymnasium[box2d]" imageio statsmodels pyvirtualdisplay tensorflow matplotlib numpy "imageio[ffmpeg]"


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: /Users/wangrunyuan/.pyenv/versions/3.11.9/bin/python -m pip install --upgrade pip


In [28]:
import time
from collections import deque, namedtuple

import numpy as np
import PIL.Image
import tensorflow as tf
import utils
import matplotlib.pyplot as plt
from bidirection_lunar_lander import BidirectionalLunarLander

# from pyvirtualdisplay import Display
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.losses import MSE
from tensorflow.keras.optimizers import Adam

In [29]:
def navie_inverted_controller(obs, phase, sign):
    x = obs[0]
    y = obs[1]
    vx = obs[2]
    vy = obs[3]
    theta = obs[4]
    omega = obs[5]

    # angle wrap to [-pi, pi]
    theta_wrapped = np.arctan2(np.sin(theta), np.cos(theta))
    theta_abs = abs(theta_wrapped)

    side = 0
    main = 0
    if phase == 1:
        if y > 1.4:
            return np.array([0, 0], dtype=np.float32), phase
            
        if theta_abs < 1.9:
            side = -0.6 * sign
            main = 0.8
        elif theta_abs > 1.9 and abs(omega) > 0.1 :
            side = 1 * sign
            main = -1
        else:
            phase = 2

    if phase == 2:
        if y > 1.4:
            return np.array([0, 0], dtype=np.float32), phase

        main = -0.6
        if vx > 0.1:
            side = 0.5
        elif vx < -0.1:
            side = -0.5
        else:
            side = 0

    return np.array([
        np.clip(main, -1.0, 1.0),
        np.clip(side, -1.0, 1.0)
    ], dtype=np.float32), phase

def getInitSign(vx0):
    if vx0 > 0:
        return 1
    else:
        return -1

In [30]:
##### See what happens by calling navie inverted controller in 500 steps
env = BidirectionalLunarLander(continuous=True, render_mode="human")
obs, info = env.reset()

sign = getInitSign(obs[2])
phase = 1
for step in range(500):
    action, phase = navie_inverted_controller(obs, phase, sign)
    obs, reward, terminated, truncated, info = env.step(action)
    x = obs[0]
    vx = obs[2]
    theta = obs[4]
    omega = obs[5]
    theta_w = np.arctan2(np.sin(theta), np.cos(theta), )
    # print(f"\rphase: {phase} action: {action} x: {x}, vx: {vx} theta: {theta_w}, omega: {omega}")

    if terminated or truncated:
        print(f"Episode ended: {step}")
        break

env.close

phase: 1 action: [0. 0.] x: 0.01139364205300808, vx: 0.5762322545051575 theta: -0.013052860274910927, omega: -0.12918056547641754
phase: 1 action: [0. 0.] x: 0.017090892419219017, vx: 0.5762525200843811 theta: -0.019507573917508125, omega: -0.12910626828670502
phase: 1 action: [0. 0.] x: 0.022788237780332565, vx: 0.576271653175354 theta: -0.02596144564449787, omega: -0.12908947467803955
phase: 1 action: [0. 0.] x: 0.028485869988799095, vx: 0.576290488243103 theta: -0.032414279878139496, omega: -0.1290690004825592
phase: 1 action: [0. 0.] x: 0.03418369218707085, vx: 0.5763095617294312 theta: -0.038866087794303894, omega: -0.12904825806617737
phase: 1 action: [0. 0.] x: 0.03988180309534073, vx: 0.576328456401825 theta: -0.04531687870621681, omega: -0.12902779877185822
phase: 1 action: [0. 0.] x: 0.04558000713586807, vx: 0.576347291469574 theta: -0.05176663398742676, omega: -0.1290072351694107
phase: 1 action: [0. 0.] x: 0.0512784942984581, vx: 0.5763660669326782 theta: -0.058215379714965

<bound method LunarLander.close of <bidirection_lunar_lander.BidirectionalLunarLander object at 0x165f06c10>>

In [31]:
def collect_one_episode(env, max_steps=500):
    obs, info = env.reset()
    sign = getInitSign(obs[2])
    phase = 1

    episode = []

    for t in range(max_steps):
        action, phase = navie_inverted_controller(obs, phase, sign)

        episode.append({
            "obs": obs.copy(),
            "action": action.copy(),
            "phase": phase
        })

        obs, reward, terminated, truncated, info = env.step(action)

        if terminated or truncated:
            break

    return episode


In [32]:
def collect_expert_episodes(
    env,
    num_episodes=300,
    min_length=30,
    require_inverted=True
):
    expert_episodes = []

    for ep in range(num_episodes):
        episode = collect_one_episode(env)

        if len(episode) < min_length:
            continue

        if require_inverted:
            phases = [step["phase"] for step in episode]
            if 2 not in phases:
                continue

        expert_episodes.append([
            {
                "obs": step["obs"],
                "action": step["action"]
            }
            for step in episode
        ])

        print(
            f"Accepted episode {len(expert_episodes)} "
            f"(len={len(episode)})"
        )

    print(f"\nCollected {len(expert_episodes)} expert episodes")
    return expert_episodes


In [33]:
def save_expert_episodes(expert_episodes, path="expert_lander.npz"):
    obs = []
    actions = []
    episode_lens = []

    for ep in expert_episodes:
        episode_lens.append(len(ep))
        for step in ep:
            obs.append(step["obs"])
            actions.append(step["action"])

    obs = np.array(obs, dtype=np.float32)
    actions = np.array(actions, dtype=np.float32)
    episode_lens = np.array(episode_lens, dtype=np.int32)

    np.savez(
        path,
        obs=obs,
        actions=actions,
        episode_lens=episode_lens
    )

    print(
        f"Saved {len(episode_lens)} episodes, "
        f"{len(obs)} total steps"
    )

In [34]:
env = BidirectionalLunarLander(continuous=True)
dataset = collect_expert_episodes(env, num_episodes=1000, min_length=50)
save_expert_episodes(dataset)

Accepted episode 1 (len=133)
Accepted episode 2 (len=195)
Accepted episode 3 (len=331)
Accepted episode 4 (len=168)
Accepted episode 5 (len=210)
Accepted episode 6 (len=135)
Accepted episode 7 (len=279)
Accepted episode 8 (len=181)
Accepted episode 9 (len=180)
Accepted episode 10 (len=165)
Accepted episode 11 (len=182)
Accepted episode 12 (len=186)
Accepted episode 13 (len=244)
Accepted episode 14 (len=336)
Accepted episode 15 (len=195)
Accepted episode 16 (len=230)
Accepted episode 17 (len=445)
Accepted episode 18 (len=259)
Accepted episode 19 (len=148)
Accepted episode 20 (len=211)
Accepted episode 21 (len=193)
Accepted episode 22 (len=212)
Accepted episode 23 (len=265)
Accepted episode 24 (len=189)
Accepted episode 25 (len=258)
Accepted episode 26 (len=178)
Accepted episode 27 (len=271)
Accepted episode 28 (len=228)
Accepted episode 29 (len=193)
Accepted episode 30 (len=173)
Accepted episode 31 (len=316)
Accepted episode 32 (len=449)
Accepted episode 33 (len=223)
Accepted episode 34

In [42]:
# Sanity check
data = np.load("expert_lander.npz")

obs_all = data["obs"]
actions_all = data["actions"]
episode_lens = data["episode_lens"]

env = BidirectionalLunarLander(
    continuous=True,
    render_mode="human"
)

ep_id = np.random.randint(len(episode_lens))

start = sum(episode_lens[:ep_id])
end = start + episode_lens[ep_id]

obs, _ = env.reset()

for t in range(start, end):
    obs, _, terminated, truncated, _ = env.step(actions_all[t])
    if terminated or truncated:
        break


In [43]:
data = np.load("expert_lander.npz")

obs_all = data["obs"]          # shape: (N, 8)
actions_all = data["actions"]  # shape: (N, 2)
episode_lens = data["episode_lens"]

dataset = tf.data.Dataset.from_tensor_slices(
    (obs_all.astype(np.float32), actions_all.astype(np.float32))
)

dataset = dataset.shuffle(10000).batch(256).prefetch(tf.data.AUTOTUNE)


In [44]:
## Behavior cloning
def build_bc_policy(obs_dim=8, act_dim=2):
    inputs = Input(shape=(obs_dim,))
    x = Dense(128, activation="relu")(inputs)
    x = Dense(128, activation="relu")(x)
    outputs = Dense(act_dim, activation="tanh")(x)

    model = Model(inputs, outputs)
    return model

In [45]:
policy = build_bc_policy()

policy.compile(
    optimizer=Adam(learning_rate=3e-4),
    loss="mse"
)

policy.fit(
    dataset,
    epochs=100
)

Epoch 1/100
874/874 ━━━━━━━━━━━━━━━━━━━━ 1s 479us/step - loss: 0.0986
Epoch 2/100
874/874 ━━━━━━━━━━━━━━━━━━━━ 0s 431us/step - loss: 0.0483
Epoch 3/100
874/874 ━━━━━━━━━━━━━━━━━━━━ 0s 434us/step - loss: 0.0408
Epoch 4/100
874/874 ━━━━━━━━━━━━━━━━━━━━ 0s 434us/step - loss: 0.0363
Epoch 5/100
874/874 ━━━━━━━━━━━━━━━━━━━━ 0s 505us/step - loss: 0.0327
Epoch 6/100
874/874 ━━━━━━━━━━━━━━━━━━━━ 0s 436us/step - loss: 0.0300
Epoch 7/100
874/874 ━━━━━━━━━━━━━━━━━━━━ 0s 451us/step - loss: 0.0281
Epoch 8/100
874/874 ━━━━━━━━━━━━━━━━━━━━ 0s 482us/step - loss: 0.0267
Epoch 9/100
874/874 ━━━━━━━━━━━━━━━━━━━━ 0s 446us/step - loss: 0.0255
Epoch 10/100
874/874 ━━━━━━━━━━━━━━━━━━━━ 0s 446us/step - loss: 0.0244
Epoch 11/100
874/874 ━━━━━━━━━━━━━━━━━━━━ 0s 439us/step - loss: 0.0235
Epoch 12/100
874/874 ━━━━━━━━━━━━━━━━━━━━ 0s 453us/step - loss: 0.0230
Epoch 13/100
874/874 ━━━━━━━━━━━━━━━━━━━━ 0s 464us/step - loss: 0.0222
Epoch 14/100
874/874 ━━━━━━━━━━━━━━━━━━━━ 0s 435us/step - loss: 0.0217
Epoch 15/100
87

In [60]:
env = BidirectionalLunarLander(
    continuous=True,
    render_mode="human"
)

obs, _ = env.reset()

for t in range(500):
    obs_batch = obs.reshape(1, -1).astype(np.float32)
    action = policy(obs_batch, training=False).numpy()[0]

    obs, _, terminated, truncated, _ = env.step(action)
    if terminated or truncated:
        break

In [61]:
class ExpertController:
    def __init__(self):
        self.phase = 0

    def reset(self, vx0):
        self.phase = 0
        self.sign = getInitSign(vx0)

    def act(self, obs):
        action, self.phase = navie_inverted_controller(obs, self.phase, self.sign)
        return action

In [62]:
def dagger_rollout(
    env,
    policy,
    expert,
    max_steps=500
):
    obs, _ = env.reset()
    expert.reset(obs[2])

    rollout_data = []

    for t in range(max_steps):
        # learner action
        obs_batch = obs.reshape(1, -1).astype(np.float32)
        learner_action = policy(obs_batch, training=False).numpy()[0]

        # expert label
        expert_action = expert.act(obs)

        rollout_data.append({
            "obs": obs.copy(),
            "action": expert_action.copy()
        })

        obs, _, terminated, truncated, _ = env.step(learner_action)

        if terminated or truncated:
            break

    return rollout_data

In [63]:
def aggregate_dataset(dataset, new_data):
    for step in new_data:
        dataset.append((
            step["obs"].astype(np.float32),
            step["action"].astype(np.float32)
        ))
        
dataset = list(zip(obs_all, actions_all))

In [64]:
def train_policy_tf(policy, dataset, epochs=5):
    obs = np.array([d[0] for d in dataset], dtype=np.float32)
    actions = np.array([d[1] for d in dataset], dtype=np.float32)

    ds = tf.data.Dataset.from_tensor_slices((obs, actions))
    ds = ds.shuffle(10000).batch(256).prefetch(tf.data.AUTOTUNE)

    policy.fit(ds, epochs=epochs, verbose=0)

In [65]:
env = BidirectionalLunarLander(continuous=True)
expert = ExpertController()

# dataset 已由 BC 初始化
for iteration in range(10):
    print(f"\nDAgger iteration {iteration}")

    rollout_data = dagger_rollout(
        env,
        policy,
        expert,
        max_steps=300
    )

    aggregate_dataset(dataset, rollout_data)

    train_policy_tf(
        policy,
        dataset,
        epochs=5
    )


DAgger iteration 0

DAgger iteration 1

DAgger iteration 2

DAgger iteration 3

DAgger iteration 4

DAgger iteration 5

DAgger iteration 6

DAgger iteration 7

DAgger iteration 8

DAgger iteration 9


In [74]:
env = BidirectionalLunarLander(
    continuous=True,
    render_mode="human"
)

obs, _ = env.reset()

for t in range(500):
    obs_batch = obs.reshape(1, -1).astype(np.float32)
    action = policy(obs_batch, training=False).numpy()[0]

    obs, _, terminated, truncated, _ = env.step(action)
    if terminated or truncated:
        break

In [ ]:
class ExpertController:
    def __init__(self):
        self.phase = 0
        self.sign = 1

    def reset(self, vx0):
        self.phase = 0
        self.sign = getInitSign(vx0)

    def act(self, obs):
        return navie_inverted_controller(obs, self.phase, self.sign)

    def _fsm_controller(self, obs):
        x, y, vx, vy, theta, omega, _, _ = obs
        theta_wrapped = np.arctan2(np.sin(theta), np.cos(theta))
        theta_abs = abs(theta_wrapped)

        main = 0.0
        side = 0.0

        if self.phase == 0:
            if theta_abs < 1.9:
                side = -0.6
                main = 0.8
            else:
                self.phase = 1

        if self.phase == 1:
            main = -0.6
            if vx > 0.1:
                side = 0.5
            elif vx < -0.1:
                side = -0.5

        return np.array(
            [np.clip(main, -1, 1), np.clip(side, -1, 1)],
            dtype=np.float32
        )

In [ ]:
class InvertedHoverWrapper(gym.Wrapper):
    def __init__(self, env, expert, y_ref=1.5):
        super().__init__(env)
        self.expert = expert
        self.y_ref = y_ref

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.expert.reset(obs[2])

        # 用 expert 把系统带入 inverted 区域
        for _ in range(200):
            action = self.expert.act(obs)
            obs, _, terminated, truncated, _ = self.env.step(action)

            if terminated or truncated:
                obs, info = self.env.reset(**kwargs)
                self.expert.reset()

            theta_err = np.arctan2(
                np.sin(obs[4] - np.pi),
                np.cos(obs[4] - np.pi)
            )
            if abs(theta_err) < 0.3:
                break

        return obs, info

    def step(self, action):
        obs, _, terminated, truncated, info = self.env.step(action)

        reward = self.tracking_reward(obs)

        done = terminated or truncated
        if abs(obs[1]) > 3.0:
            done = True

        return obs, reward, done, False, info

    def tracking_reward(self, obs):
        x, y, vx, vy, theta, omega, _, _ = obs

        theta_err = np.arctan2(
            np.sin(theta - np.pi),
            np.cos(theta - np.pi)
        )

        r = (
            - 2.0 * theta_err**2
            - 1.5 * (y - self.y_ref)**2
            - 1.0 * vy**2
            - 0.5 * vx**2
            - 0.2 * omega**2
        )
        return r